# 14-1: Overview of Generalized Linear Models (GLMs)

**Course:** Models of Statistical Analysis (MAE) — Universidad de los Andes  
**Instructor:** Prof. Alejandra Tabares  
**Week:** 14

## 1. The GLM Framework

A **Generalized Linear Model** unifies a broad class of regression models under three components:

### 1.1 Random Component

The response variable $Y_i$ follows a distribution from the **exponential family**:

$$
f(y_i; \theta_i, \phi) = \exp\!\left[\frac{y_i\theta_i - b(\theta_i)}{a(\phi)} + c(y_i, \phi)\right]
$$

where $\theta_i$ is the canonical (natural) parameter, $\phi$ is the dispersion parameter, and $a(\cdot)$, $b(\cdot)$, $c(\cdot)$ are known functions. The mean is $\mu_i = b'(\theta_i)$ and the variance is $\text{Var}(Y_i) = b''(\theta_i)\,a(\phi)$.

### 1.2 Systematic Component

The **linear predictor** combines the covariates:

$$
\eta_i = \beta_0 + \beta_1 x_{i1} + \cdots + \beta_p x_{ip} = \mathbf{x}_i^\top \boldsymbol{\beta}
$$

### 1.3 Link Function

The **link function** $g$ connects the mean $\mu_i$ to the linear predictor:

$$
g(\mu_i) = \eta_i \qquad \Longleftrightarrow \qquad \mu_i = g^{-1}(\eta_i)
$$

When $g$ is the canonical link, $\theta_i = \eta_i$ and the score equations take a particularly simple form.

> **Key insight:** The classical linear model is the special case where $Y_i \sim \mathcal{N}(\mu_i, \sigma^2)$ and $g$ is the identity. GLMs extend this to non-normal, non-continuous responses while keeping the linear predictor.

## 2. Common GLM Families

| Family | Response type | Canonical link | Link function $g(\mu)$ | Variance function $V(\mu)$ | Typical use case |
|---|---|---|---|---|---|
| **Gaussian** | Continuous, unbounded | Identity | $\mu$ | $1$ | Classical regression |
| **Binomial** | Proportions / binary | Logit | $\ln\!\left(\frac{\mu}{1-\mu}\right)$ | $\mu(1-\mu)$ | Binary outcomes, success rates |
| **Poisson** | Non-negative counts | Log | $\ln(\mu)$ | $\mu$ | Event counts, rare events |
| **Gamma** | Positive continuous | Inverse | $1/\mu$ | $\mu^2$ | Skewed durations, costs |
| **Inverse Gaussian** | Positive continuous | $1/\mu^2$ | $1/\mu^2$ | $\mu^3$ | Highly skewed durations |
| **Negative Binomial** | Over-dispersed counts | Log | $\ln(\mu)$ | $\mu + \mu^2/r$ | Count data with overdispersion |

**Notes:**
- *Canonical link* is theoretically convenient but not always optimal in practice.
- For Binomial, probit ($\Phi^{-1}(\mu)$) and complementary log-log ($\ln(-\ln(1-\mu))$) are common alternatives.
- For Gamma, the log link is often preferred over the canonical inverse link for interpretability.

## 3. Fitting GLMs with `statsmodels` — Simulated Examples

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n = 300

# Shared covariates
x1 = np.random.normal(0, 1, n)
x2 = np.random.normal(0, 1, n)

# True linear predictor
eta = 1.0 + 0.8 * x1 - 0.5 * x2

# --- Gaussian response (identity link) ---
y_gaussian = eta + np.random.normal(0, 1, n)

# --- Binomial response (logit link) ---
p = 1 / (1 + np.exp(-eta))
y_binomial = np.random.binomial(1, p, n)

# --- Poisson response (log link) ---
mu_poisson = np.exp(0.5 + 0.4 * x1 - 0.3 * x2)
y_poisson = np.random.poisson(mu_poisson, n)

# --- Gamma response (log link for interpretability) ---
mu_gamma = np.exp(0.5 + 0.6 * x1 - 0.4 * x2)
shape_param = 3.0  # shape k; mean = k/rate => rate = k/mu
y_gamma = np.random.gamma(shape=shape_param, scale=mu_gamma / shape_param, size=n)

df = pd.DataFrame({'x1': x1, 'x2': x2,
                   'y_gaussian': y_gaussian,
                   'y_binomial': y_binomial,
                   'y_poisson': y_poisson,
                   'y_gamma': y_gamma})
print('Dataset shape:', df.shape)
print(df.head())

In [ ]:
# ── Fit all four GLMs ──────────────────────────────────────────────────────────

# 1. Gaussian GLM (equivalent to OLS)
glm_gaussian = smf.glm('y_gaussian ~ x1 + x2', data=df,
                        family=sm.families.Gaussian()).fit()

# 2. Binomial GLM (logistic regression)
glm_binomial = smf.glm('y_binomial ~ x1 + x2', data=df,
                        family=sm.families.Binomial()).fit()

# 3. Poisson GLM (log link)
glm_poisson = smf.glm('y_poisson ~ x1 + x2', data=df,
                       family=sm.families.Poisson()).fit()

# 4. Gamma GLM (log link)
glm_gamma = smf.glm('y_gamma ~ x1 + x2', data=df,
                     family=sm.families.Gamma(link=sm.families.links.Log())).fit()

# Print coefficient tables
for name, model in [('Gaussian', glm_gaussian), ('Binomial', glm_binomial),
                     ('Poisson', glm_poisson), ('Gamma', glm_gamma)]:
    print(f'\n{'='*55}')
    print(f'  {name} GLM — coefficient estimates')
    print(f'{'='*55}')
    print(model.summary2().tables[1].to_string())

## 4. How to Choose a GLM Family

Selecting the right GLM family involves both substantive reasoning and empirical checks:

### Step 1 — Look at the response variable

| Response type | First candidate family |
|---|---|
| Continuous, approximately symmetric | **Gaussian** |
| Binary (0/1) or proportion | **Binomial** |
| Non-negative integer (count) | **Poisson** |
| Positive continuous, right-skewed | **Gamma** |
| Positive continuous, highly skewed | **Inverse Gaussian** |

### Step 2 — Check the mean-variance relationship

Plot the group means against the group variances (or residuals). If variance grows with the mean:
- Proportionally → Poisson
- As $\mu^2$ → Gamma
- Faster than $\mu^2$ → consider Negative Binomial or quasi-Poisson

### Step 3 — Compare candidate models

Use **AIC** (Akaike Information Criterion) to compare models with the same response variable but different families or link functions:
$$
\text{AIC} = -2\,\ell(\hat{\boldsymbol{\beta}}) + 2p
$$
Lower AIC is better. Do not compare AIC across different response transformations.

### Step 4 — Inspect residuals

Use **deviance residuals** or **Pearson residuals** plotted against fitted values to check for patterns.

In [ ]:
# ── AIC comparison table ───────────────────────────────────────────────────────

# We fit competing link functions for the Gamma model to illustrate AIC selection
glm_gamma_inv = smf.glm('y_gamma ~ x1 + x2', data=df,
                          family=sm.families.Gamma(
                              link=sm.families.links.InversePower())).fit()
glm_gamma_id  = smf.glm('y_gamma ~ x1 + x2', data=df,
                          family=sm.families.Gamma(
                              link=sm.families.links.Identity())).fit()

aic_table = pd.DataFrame({
    'Model': [
        'Gaussian (identity)',
        'Binomial (logit)',
        'Poisson (log)',
        'Gamma (log link)',
        'Gamma (inverse link)',
        'Gamma (identity link)'
    ],
    'Response': ['y_gaussian', 'y_binomial', 'y_poisson',
                 'y_gamma', 'y_gamma', 'y_gamma'],
    'AIC': [
        glm_gaussian.aic,
        glm_binomial.aic,
        glm_poisson.aic,
        glm_gamma.aic,
        glm_gamma_inv.aic,
        glm_gamma_id.aic
    ],
    'Deviance': [
        glm_gaussian.deviance,
        glm_binomial.deviance,
        glm_poisson.deviance,
        glm_gamma.deviance,
        glm_gamma_inv.deviance,
        glm_gamma_id.deviance
    ],
    'df_resid': [
        glm_gaussian.df_resid,
        glm_binomial.df_resid,
        glm_poisson.df_resid,
        glm_gamma.df_resid,
        glm_gamma_inv.df_resid,
        glm_gamma_id.df_resid
    ]
})

aic_table['AIC'] = aic_table['AIC'].round(2)
aic_table['Deviance'] = aic_table['Deviance'].round(4)
print('\nAIC Comparison Table')
print('=' * 70)
print(aic_table.to_string(index=False))

# Among the Gamma models, log link wins (lowest AIC)
gamma_aic = aic_table[aic_table['Response'] == 'y_gamma'].copy()
best = gamma_aic.loc[gamma_aic['AIC'].idxmin(), 'Model']
print(f'\nBest Gamma link by AIC: {best}')

In [ ]:
# ── Residual plots for GLM diagnostics ────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('GLM Deviance Residuals vs Fitted Values', fontsize=14, y=1.01)

models_plot = [
    (glm_gaussian, 'Gaussian (identity)', 'steelblue'),
    (glm_binomial, 'Binomial (logit)',    'darkorange'),
    (glm_poisson,  'Poisson (log)',       'seagreen'),
    (glm_gamma,    'Gamma (log)',         'crimson'),
]

for ax, (model, title, color) in zip(axes.flatten(), models_plot):
    fitted = model.fittedvalues
    resid  = model.resid_deviance
    ax.scatter(fitted, resid, alpha=0.4, s=18, color=color)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Fitted values', fontsize=10)
    ax.set_ylabel('Deviance residuals', fontsize=10)
    ax.set_title(title, fontsize=11)

plt.tight_layout()
plt.show()

## 5. Deviance and Goodness of Fit in GLMs

### Deviance

The **deviance** generalizes the residual sum of squares (RSS) to GLMs:

$$
D = 2\left[\ell(\text{saturated model}) - \ell(\hat{\boldsymbol{\beta}})\right]
$$

- The **saturated model** has one parameter per observation ($\hat{\mu}_i = y_i$), achieving the maximum possible log-likelihood.
- For a Gaussian GLM, $D = \text{RSS}/\sigma^2$ — identical to the classical F-test residual SS.
- For Poisson: $D = 2\sum_i \left[y_i \ln(y_i/\hat{\mu}_i) - (y_i - \hat{\mu}_i)\right]$

### Null and Residual Deviance

| Quantity | Definition | Degrees of freedom |
|---|---|---|
| **Null deviance** | Deviance of intercept-only model | $n - 1$ |
| **Residual deviance** | Deviance of fitted model | $n - p - 1$ |
| **Deviance explained** | Null $-$ Residual | $p$ |

### Pearson $\chi^2$ Statistic

$$
X^2 = \sum_{i=1}^n \frac{(y_i - \hat{\mu}_i)^2}{V(\hat{\mu}_i)}
$$

Both $D$ and $X^2$ follow approximately $\chi^2_{n-p-1}$ under the model — but only in large samples and for continuous responses.

### Overdispersion

For Poisson/Binomial GLMs, the **dispersion parameter** $\phi$ is fixed at 1. If $D/(n-p-1) \gg 1$, the data exhibit **overdispersion**. Remedies:
- Use **quasi-Poisson** or **quasi-Binomial** (estimate $\phi$ from data)
- Switch to **Negative Binomial** regression

### Pseudo-$R^2$

$$
R^2_{\text{McFadden}} = 1 - \frac{\ell(\hat{\boldsymbol{\beta}})}{\ell(\boldsymbol{\beta}_0)}
$$

Useful for relative comparisons, not as an absolute measure.